# Phase 2B.0 - Preparation (Kaggle)

Freeze all subsets and package the Phase 2A baseline inputs. No model API calls are made.


In [ ]:
from pathlib import Path
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='ada57f8ed5dcdb0d6824a19f0bc80f5323e744f9'
HF_ARTIFACT_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-locked-v2'
HF_ARTIFACT_REVISION='locked-bge-m3-512-64-deduplicated-v2'
HF_ARTIFACT_FILENAME='artifacts/locked-bge-m3-512-64-deduplicated-v2/locked-bge-m3-512-64-deduplicated-v2.zip'
HF_ARTIFACT_SHA256='fc5d67b7acf6e8be0205ce00b8069b3b6c8dcce853f8671f2feb3887b2707a24'
PLATFORM='kaggle'
RUN_ID='phase2b_0_preparation'
PREPARATION_BUNDLE_PATH=''  # Required except in Phase 2B.0; Drive path on Colab or attached Kaggle input.
BASELINE_RESULTS_PATH=''  # Optional local ZIP/directory override; otherwise download from private HF.
HF_BASELINE_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-experiments'
HF_BASELINE_REVISION='bd83647c784d7f7d466bbd954fe7d35b5252a2f5'  # phase2a-baseline-deduplicated-v2
HF_BASELINE_FILENAME='phase2a/baseline-deduplicated-v2/phase2_e2e_baseline_full_results.zip'
HF_BASELINE_SHA256='044c913d20942c2dc8c1dcfa39544a7baf082f386b6d5f934e01c9c7feaf6017'
RESTORE_CHECKPOINT_PATH=''  # Optional checkpoint from the same notebook only.
EXECUTE_API_CALLS=False  # Set True only for an approved execution notebook.
GEMINI_SECRET_NAME='GEMINI_API_KEY_1'  # Give each Colab prompt/depth notebook its own key.
UPSTREAM_DECISION_PATH=''  # Optional JSON decision record; retained in provenance.
RUN_STAGE=RUN_ID
PROMPT_ID=None
PROMPT_FINALISTS=[]
FINALIST_CONFIG=None
FINALIST_CONFIGS=[]
LOCKED_WINNER=None
HELDOUT_APPROVED=False
CONTEXT_DEPTH=None
SEED=42
SCREENING_QUESTIONS=80
JUDGE_CALIBRATION_QUESTIONS=20
FINAL_HELDOUT_ARTICLES=50
GENERATOR_MODEL='gemini-3.1-flash-lite'
GENERATOR_REASONING_EFFORT='minimal'
GENERATOR_MAX_TOKENS=512
GENERATOR_MIN_INTERVAL_SECONDS=4.2
JUDGE_MODEL='accounts/fireworks/models/glm-5p3-flash'
JUDGE_REASONING_EFFORT='low'
JUDGE_MAX_TOKENS=2048
GENERATOR_INPUT_PER_MILLION_USD=0.25
GENERATOR_OUTPUT_PER_MILLION_USD=1.50
JUDGE_INPUT_PER_MILLION_USD=0.15
JUDGE_OUTPUT_PER_MILLION_USD=0.50
TOP_K=20
RERANK_TOP_N=5


## 1. Environment and immutable inputs

Set only the configuration values at the top. Every run validates the locked artifact, preparation bundle, subset hashes, and repository commit.


In [ ]:
import hashlib, json, os, shutil, string, subprocess, sys, time, zipfile
KAGGLE_INPUT=Path('/kaggle/input'); RUNTIME_ROOT=Path('/kaggle/working')
from kaggle_secrets import UserSecretsClient
secrets=UserSecretsClient()
def optional_secret(name):
    try: return secrets.get_secret(name) or ''
    except Exception: return ''
PROJECT_ROOT=RUNTIME_ROOT/'Text-Mining---NewsQA-RAG'
WORK_ROOT=RUNTIME_ROOT/RUN_ID
DATA_ROOT=WORK_ROOT/'data'; INDEX_ROOT=WORK_ROOT/'index'; BASELINE_ROOT=WORK_ROOT/'baseline'
RUNS_ROOT=WORK_ROOT/'runs'; IDS_ROOT=WORK_ROOT/'question_ids'; PROMPT_ROOT=WORK_ROOT/'prompts'; RESULTS=WORK_ROOT/'results'; LOGS=WORK_ROOT/'logs'
checkpoint_input=Path(RESTORE_CHECKPOINT_PATH) if RESTORE_CHECKPOINT_PATH else None
if checkpoint_input and checkpoint_input.exists():
    WORK_ROOT.mkdir(parents=True,exist_ok=True); shutil.unpack_archive(checkpoint_input,WORK_ROOT); print('Restored checkpoint:',checkpoint_input)
for path in [DATA_ROOT,INDEX_ROOT,BASELINE_ROOT,RUNS_ROOT,IDS_ROOT,PROMPT_ROOT,RESULTS,LOGS]: path.mkdir(parents=True,exist_ok=True)
assert not REPO_COMMIT.startswith('SET_TO_'), 'Pin REPO_COMMIT after committing the split notebooks'
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
GENERATOR_API_KEY=optional_secret(GEMINI_SECRET_NAME); JUDGE_API_KEY=optional_secret('FIREWORKS_API_KEY'); HF_TOKEN=optional_secret('HF_TOKEN')
os.environ.update({'HF_HOME':str(RUNTIME_ROOT/'hf_cache'),'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1','LANGCHAIN_TRACING_V2':'false','LANGSMITH_TRACING':'false'})
if EXECUTE_API_CALLS:
    assert GENERATOR_API_KEY, f'Configure the secret named {GEMINI_SECRET_NAME}'
    assert JUDGE_API_KEY, 'Configure FIREWORKS_API_KEY'
print('Run:',RUN_ID,'| platform:',PLATFORM,'| API execution:',EXECUTE_API_CALLS)


In [ ]:
import pandas as pd, yaml
from IPython.display import display
def sha256_file(path,block_size=1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''): digest.update(block)
    return digest.hexdigest()
def write_json(path,value): Path(path).write_text(json.dumps(value,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def load_jsonl(path): return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
def write_jsonl(path,records): Path(path).write_text(''.join(json.dumps(row,sort_keys=True)+'\n' for row in records),encoding='utf-8')
def write_checkpoint():
    checkpoint=RUNTIME_ROOT/f'{RUN_ID}_checkpoint.zip'; temporary=checkpoint.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for name in ['baseline','index','runs','question_ids','prompts','results','logs','heldout_trace']:
            root=WORK_ROOT/name
            if root.exists():
                for path in root.rglob('*'):
                    if path.is_file(): archive.write(path,path.relative_to(WORK_ROOT))
    temporary.replace(checkpoint); print('Checkpoint:',checkpoint,round(checkpoint.stat().st_size/2**20,1),'MiB',flush=True); return checkpoint
def run_command(command,label,env_overrides=None):
    command=[str(value) for value in command]; log_path=LOGS/f'{label}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('$',' '.join(command),flush=True); print('Log:',log_path,flush=True)
    with log_path.open('w',encoding='utf-8') as log:
        env=os.environ.copy(); env.update(env_overrides or {})
        process=subprocess.Popen(command,cwd=PROJECT_ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        for line in process.stdout: print(line,end='',flush=True); log.write(line); log.flush()
        code=process.wait()
    if code: write_checkpoint(); raise subprocess.CalledProcessError(code,command)
    return log_path
def stable_key(seed,*parts): return hashlib.sha256((':'.join([str(seed),*map(str,parts)])).encode()).hexdigest()
def configuration_fingerprint(prompt_id,context_depth,ids):
    payload={'baseline_fingerprint':baseline_manifest['run_fingerprint'],'prompt_id':prompt_id,'prompt':prompt_registry[prompt_id]['system_prompt'],'context_depth':context_depth,'question_ids':list(ids),'generator_model':GENERATOR_MODEL,'generator_reasoning_effort':GENERATOR_REASONING_EFFORT}
    return hashlib.sha256(json.dumps(payload,sort_keys=True,separators=(',',':')).encode()).hexdigest()
def article_balanced_order(rows,seed):
    buckets={}
    for row in rows: buckets.setdefault(row['article_key'],[]).append(row)
    for article in buckets: buckets[article].sort(key=lambda row:stable_key(seed,row['question_id']))
    articles=sorted(buckets,key=lambda article:stable_key(seed,article)); ordered=[]
    while any(buckets.values()):
        for article in articles:
            if buckets[article]: ordered.append(buckets[article].pop(0))
    return ordered
def question_type(question):
    first=question.strip().lower().split(maxsplit=1)[0].strip(string.punctuation) if question.strip() else 'other'
    return first if first in {'who','what','when','where','why','how','which'} else ('yes_no' if first in {'is','are','was','were','do','does','did','has','have','had','can','could','will','would'} else 'other')
def proportional_stratified_sample(rows,n,seed):
    assert 0<n<=len(rows); groups={}
    for row in rows: groups.setdefault((row['question_type'],row['gold_in_top5']),[]).append(row)
    quotas={key:min(len(group),int(n*len(group)/len(rows))) for key,group in groups.items()}
    while sum(quotas.values())<n:
        candidates=[key for key,group in groups.items() if quotas[key]<len(group)]
        key=max(candidates,key=lambda value:(n*len(groups[value])/len(rows)-quotas[value],stable_key(seed,*value)))
        quotas[key]+=1
    selected=[]
    for key,group in groups.items():
        ordered=article_balanced_order(group,seed)
        selected.extend(ordered[:quotas[key]])
    return [row['question_id'] for row in sorted(selected,key=lambda row:stable_key(seed,row['question_id']))]


## 2. Load validated inputs

Phase 2B.0 downloads the private Phase 2A baseline by exact HF commit and SHA-256. It also downloads and verifies the separate locked corpus artifact.


In [ ]:
from huggingface_hub import hf_hub_download
artifact_root=DATA_ROOT/'locked-bge-m3-512-64-deduplicated-v2'
if not (artifact_root/'bundle_manifest.json').exists():
    bundle=hf_hub_download(repo_id=HF_ARTIFACT_REPO_ID,repo_type='dataset',revision=HF_ARTIFACT_REVISION,filename=HF_ARTIFACT_FILENAME,token=HF_TOKEN or None)
    assert sha256_file(bundle)==HF_ARTIFACT_SHA256
    shutil.unpack_archive(bundle,artifact_root)
bundle_manifest=json.loads((artifact_root/'bundle_manifest.json').read_text())
assert bundle_manifest['statistics']['chunks']==22766 and bundle_manifest['statistics']['resolved_questions']==1152
for relative,record in bundle_manifest['artifacts'].items():
    path=artifact_root/relative; assert path.exists() and path.stat().st_size==record['bytes'] and sha256_file(path)==record['sha256'], relative
testset=artifact_root/'testset_resolved.jsonl'; chunks=artifact_root/'chunks.jsonl'; sparse_index=artifact_root/'bge_m3_sparse.pkl'
baseline_input=Path(BASELINE_RESULTS_PATH) if BASELINE_RESULTS_PATH else None
if baseline_input is None:
    assert HF_TOKEN, 'Configure the read-only HF_TOKEN Kaggle secret for the private Phase 2 results repo'
    baseline_input=Path(hf_hub_download(repo_id=HF_BASELINE_REPO_ID,repo_type='dataset',revision=HF_BASELINE_REVISION,filename=HF_BASELINE_FILENAME,token=HF_TOKEN))
    assert sha256_file(baseline_input)==HF_BASELINE_SHA256, 'Phase 2A baseline archive checksum mismatch'
if baseline_input.is_file(): shutil.unpack_archive(baseline_input,BASELINE_ROOT)
else: shutil.copytree(baseline_input,BASELINE_ROOT,dirs_exist_ok=True)
manifest_candidates=list(BASELINE_ROOT.rglob('run_manifest.json')); assert manifest_candidates, 'Baseline run_manifest.json is missing'
baseline_manifest_path=next(path for path in manifest_candidates if len(json.loads(path.read_text()).get('inputs',{}).get('question_ids',[]))==281)
baseline_dir=baseline_manifest_path.parent; baseline_manifest=json.loads(baseline_manifest_path.read_text())
baseline_retrievals=baseline_dir/'retrievals.jsonl'; baseline_predictions=baseline_dir/'predictions.jsonl'; baseline_judges=baseline_dir/'judge_results.jsonl'
for path in [baseline_retrievals,baseline_predictions,baseline_judges]: assert path.exists(), path
assert baseline_manifest['inputs']['generator_model']==GENERATOR_MODEL
assert baseline_manifest['inputs'].get('generator_reasoning_effort')==GENERATOR_REASONING_EFFORT
assert baseline_manifest['inputs']['rerank_top_n']==RERANK_TOP_N
print('Imported baseline:',baseline_dir); print('Development questions:',len(baseline_manifest['inputs']['question_ids']))


In [ ]:
config=yaml.safe_load((PROJECT_ROOT/'configs/config.yaml').read_text())
config['chunking'].update({'strategy':'recursive','chunk_size':512,'chunk_overlap':64})
config['llm'].update({'model':GENERATOR_MODEL,'temperature':0.0,'max_tokens':GENERATOR_MAX_TOKENS,'reasoning_effort':GENERATOR_REASONING_EFFORT})
config['retrieval'].update({'retriever':'sparse','top_k':TOP_K})
config['retrieval']['sparse'].update({'method':'bge-m3','model':'BAAI/bge-m3','device':'cuda'})
config['retrieval']['reranker'].update({'enabled':True,'type':'cross-encoder','model':'BAAI/bge-reranker-large','top_n':RERANK_TOP_N,'batch_size':8,'device':'cuda'})
config_path=INDEX_ROOT/'phase2b_config.yaml'; config_path.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
profile=json.loads((artifact_root/'deduplication/deduplicated.variant.json').read_text())
config_hash=hashlib.sha256(json.dumps(config,sort_keys=True,separators=(',',':')).encode()).hexdigest()
profile['pipeline'].update({'config_path':str(config_path),'config_sha256':config_hash})
profile['database'].update({'indexed':False,'chunk_count':22766})
profile['artifacts']['chunks']={'path':str(chunks),'bytes':chunks.stat().st_size,'sha256':sha256_file(chunks)}
profile['artifacts']['testset_resolved']={'path':str(testset),'bytes':testset.stat().st_size,'sha256':sha256_file(testset)}
profile['artifacts']['bm25']={'path':str(sparse_index),'bytes':sparse_index.stat().st_size,'sha256':sha256_file(sparse_index)}
profile_path=INDEX_ROOT/'phase2b_variant.json'; write_json(profile_path,profile)
prompt_registry=yaml.safe_load((PROJECT_ROOT/'configs/experiments/phase2_generation_prompts.yaml').read_text())['prompts']
from newsqa_rag.llm import OpenAILLM
assert set(prompt_registry)=={'p0','p1','p2','p3'} and prompt_registry['p0']['system_prompt']==OpenAILLM.DEFAULT_SYSTEM_PROMPT
for prompt_id,record in prompt_registry.items(): (PROMPT_ROOT/f'{prompt_id}.txt').write_text(record['system_prompt'],encoding='utf-8')
display(pd.DataFrame([{'prompt_id':key,'name':value['name'],'hypothesis':value['hypothesis']} for key,value in prompt_registry.items()]))


In [ ]:
test_rows={row['question_id']:row for row in load_jsonl(testset)}
trace_records={row['question_id']:row for row in load_jsonl(baseline_retrievals)}
development_ids=baseline_manifest['inputs']['question_ids']; assert len(development_ids)==281 and len(set(development_ids))==281
assert all(qid in test_rows and qid in trace_records and trace_records[qid]['status']=='success' for qid in development_ids)
development_articles={test_rows[qid]['article_key'] for qid in development_ids}; assert len(development_articles)==50
heldout_pool_articles=sorted({row['article_key'] for row in test_rows.values()}-development_articles); assert len(heldout_pool_articles)==150
heldout_pool_ids=[qid for qid,row in test_rows.items() if row['article_key'] in set(heldout_pool_articles)]; assert len(heldout_pool_ids)==871
heldout_articles=sorted(heldout_pool_articles,key=lambda article:stable_key(SEED+4,article))[:FINAL_HELDOUT_ARTICLES]
heldout_article_set=set(heldout_articles); heldout_ids=[qid for qid,row in test_rows.items() if row['article_key'] in heldout_article_set]
heldout_reserve_ids=[qid for qid in heldout_pool_ids if qid not in set(heldout_ids)]
assert len(heldout_articles)==50 and len(heldout_ids)==284 and len(heldout_reserve_ids)==587
sampling_rows=[]
for qid in development_ids:
    row=test_rows[qid]; ranked=trace_records[qid]['trace']['retrieved_ids'][:5]
    sampling_rows.append({'question_id':qid,'article_key':row['article_key'],'question_type':question_type(row['question']),'gold_in_top5':bool(set(row['relevant_chunk_ids'])&set(ranked))})
smoke_ids=proportional_stratified_sample(sampling_rows,5,SEED+1)
screening_ids=proportional_stratified_sample(sampling_rows,SCREENING_QUESTIONS,SEED+2)
screening_rows=[row for row in sampling_rows if row['question_id'] in set(screening_ids)]
judge_calibration_ids=proportional_stratified_sample(screening_rows,JUDGE_CALIBRATION_QUESTIONS,SEED+3)
subsets={'smoke':smoke_ids,'screening':screening_ids,'judge_calibration':judge_calibration_ids,'development':development_ids,'heldout':heldout_ids,'heldout_reserve':heldout_reserve_ids}
for name,ids in subsets.items(): write_json(IDS_ROOT/f'{name}.json',ids)
subset_manifest={'schema_version':1,'baseline_artifact':{'repo_id':HF_BASELINE_REPO_ID,'revision':HF_BASELINE_REVISION,'filename':HF_BASELINE_FILENAME,'sha256':HF_BASELINE_SHA256},'seed':SEED,'source_baseline_fingerprint':baseline_manifest['run_fingerprint'],'counts':{name:len(ids) for name,ids in subsets.items()},'sha256':{name:sha256_file(IDS_ROOT/f'{name}.json') for name in subsets},'heldout_selection':{'method':'seeded_article_sample','seed':SEED+4,'articles':len(heldout_articles),'questions':len(heldout_ids),'article_ids':heldout_articles},'heldout_outputs_accessed_for_selection':False}
write_json(RESULTS/'subset_manifest.json',subset_manifest); display(pd.DataFrame([{'subset':name,'questions':len(ids)} for name,ids in subsets.items()]))
display(pd.DataFrame(sampling_rows).groupby(['question_type','gold_in_top5']).size().rename('development').reset_index())


## 4. Export

Download or retain both the result bundle and checkpoint. Later stages must be configured from reviewed result artifacts, never by changing subset IDs.


In [ ]:
bundle=RUNTIME_ROOT/'phase2b_preparation_bundle.zip'
temporary=bundle.with_suffix('.zip.tmp')
with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ['baseline','index','question_ids','prompts','results']:
        root=WORK_ROOT/name
        for path in root.rglob('*'):
            if path.is_file(): archive.write(path,path.relative_to(WORK_ROOT))
temporary.replace(bundle)
print('Preparation bundle:',bundle,round(bundle.stat().st_size/2**20,1),'MiB')
